In [ ]:
import torch
print(torch.cuda.is_available())      # doit afficher True
print(torch.cuda.get_device_name(0))  # doit afficher Tesla T4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Créer la structure dans ton Drive
os.makedirs('/content/drive/MyDrive/geoscanner/dataset', exist_ok=True)
os.makedirs('/content/drive/MyDrive/geoscanner/models',  exist_ok=True)

# Vérification
print("✅ Dossiers créés :")
print(os.listdir('/content/drive/MyDrive/geoscanner'))

In [ ]:
!pip install kaggle -q

In [ ]:
from google.colab import files

# Uploader ton kaggle.json
files.upload()  # sélectionne le fichier kaggle.json téléchargé

In [ ]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle configuré !")

In [ ]:
!kaggle datasets download -d stealthtechnologies/rock-classification \
    -p /content/drive/MyDrive/geoscanner/dataset/

print("✅ Dataset téléchargé !")

In [ ]:
import os

BASE = '/content/drive/MyDrive/geoscanner/dataset'

# Voir tout ce qui est dans le dossier dataset
print("Contenu de dataset/ :")
for item in os.listdir(BASE):
    print(f"  {item}")

In [ ]:
import os

BASE = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'

for split in ['train', 'valid', 'test']:
    path = os.path.join(BASE, split)
    if os.path.exists(path):
        classes = os.listdir(path)
        total_images = sum(
            len(os.listdir(os.path.join(path, cls)))
            for cls in classes
        )
        print(f"{split:6} → {len(classes)} classes — {total_images} images")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

print("✅ Imports OK")

In [ ]:
BASE       = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'
MODEL_DIR  = '/content/drive/MyDrive/geoscanner/models'
MODEL_PATH = os.path.join(MODEL_DIR, 'resnet50_best.pth')
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Device : {device}")
print(f"✅ BASE   : {BASE}")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds     = datasets.ImageFolder(os.path.join(BASE, 'train'), train_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
print("✅ train_transform original restauré")

In [ ]:
train_ds = datasets.ImageFolder(os.path.join(BASE, 'train'), train_transform)
valid_ds = datasets.ImageFolder(os.path.join(BASE, 'valid'), val_transform)
test_ds  = datasets.ImageFolder(os.path.join(BASE, 'test'),  val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Train : {len(train_ds)} images")
print(f"✅ Valid : {len(valid_ds)} images")
print(f"✅ Test  : {len(test_ds)} images")
print(f"✅ Classes : {train_ds.classes}")

In [ ]:
model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
print("✅ Modèle réinitialisé")

In [ ]:
model.fc.requires_grad_(True)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_A     = 5
best_acc     = 0.0
patience_ctr = 0
PATIENCE     = 4
history      = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — FC uniquement  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Phase A terminée — Meilleure acc : {best_acc:.2f}%")

In [ ]:
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.00001},
    {'params': model.layer4.parameters(), 'lr': 0.00005},
    {'params': model.fc.parameters(),     'lr': 0.0001},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=4, factor=0.3
)

EPOCHS_B     = 20
patience_ctr = 0
PATIENCE     = 5

print("=" * 55)
print("PHASE B — layer3+layer4+FC  lr conservateur")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")